# iter5_hall — ALL the data, pure hard-CE (owner's 'keep it simple, use everything')

**Corpus `h_all.pt` = 5,296,688 rows (64% crisis-replay)**: every successful crisis
replay IN FULL (escape + healthy 500-turn continuation = 'selfplay from a new position'),
failure prefixes (terminal 20 rows cut), all selfplay (10k/game outlier cap, first-30
temperature moves masked). Every target row's argmax == the PLAYED move.
**Recipe:** warm-start vh2, **blend 0.0** (pure hard CE), dw0, T1.0, lr 1e-4, bs 8192,
NORMAL BN, step saves every 100. Nothing else.

**Upload to `MyDrive/alphatrain/`:** `h_all.pt.gz` (308,489,045 B). Tarball v5 +
`small128_vh2.pt` already on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v5.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v5.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/h_all.pt.gz /content/h_all.pt.gz
gz=os.path.getsize('/content/h_all.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 308_489_045, f'.gz truncated! got {gz}; re-upload h_all.pt.gz'
!gunzip -t /content/h_all.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/h_all.pt.gz > /content/alphatrain/data/h_all.pt
pt=os.path.getsize('/content/alphatrain/data/h_all.pt')
assert pt == 1_202_351_351, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 6,175,263 states ({time.time()-t0:.0f}s)')
!rm /content/h_all.pt.gz
!cp {DRIVE}/small128_vh2.pt /content/alphatrain/data/
assert os.path.getsize('/content/alphatrain/data/small128_vh2.pt') == 12_067_557
!pip install -q numpy numba scipy


In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (iter5_h_all: pure hard-CE) =====
CHANNELS = 128
EPOCHS   = 8  # bulk hard-CE absorption is the one historically step-hungry mode; ladder screens protect the early region
BATCH    = 8192
LR       = 1e-4
T        = 1.0
DW       = 0
BLEND    = 0.0  # PURE hard CE on the played move
SEED     = 42
SAVE_STEPS = 100
RUN      = "iter5_h_all"
print(f'RUN={RUN} ep={EPOCHS} bs={BATCH} lr={LR} blend={BLEND}')


In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/h_all.pt \
    --resume alphatrain/data/small128_vh2.pt --warm-start \
    --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
# val is informational only — the gate is GAMEPLAY (floor-first, 5k decides).


In [ ]:
# Checkpoints already save DIRECTLY to Drive ({RUN}_ckpts/) —
# every epoch_N.pt and e{E}_s{S}.pt lands there as it is written.
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)


## Gate protocol (M5, C++ eval — download checkpoints as they save)

```bash
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/iter5a_e1_s400.pt
mv alphatrain/inference_cpp/data/policy_ts.pt alphatrain/inference_cpp/data/dagger1_e1_s400_ts.pt
cd alphatrain/inference_cpp
./build/eval --model data/dagger1_e1_s400_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
1. **Screen step checkpoints** (s100..s900 + epoch saves) at 500 seeds. The 500-seed screen is a
   **catastrophe filter ONLY** — it rejects blowups; it does NOT rank close calls (vh1 itself read
   P50 −6% at 500 seeds, then WON +4.6% at 5k — HISTORY:3375).
2. Take the 2-3 best-looking survivors (floor-first: <1000%, P5/P10) to the **5k eval**
   (`--seed-end 780000`) against vh1's bar: mean 13,080 / P50 9,323 / P5 1,222 / <1000 3.5%.
3. Expected per the MICRO-GO calibration: +3-8%% median. If it clears → name it `small128_vh2`,
   retrain the survival head on ITS backbone, re-export, then RE-GATE Engine B (own-MCTS judge)
   at the new level — the alternation plan (memory: project_small_model_distill).
4. All step checkpoints regress at 500 (catastrophically) → recheck mix %% and label sanity
   before ANY recipe surgery (HISTORY 178 lesson: measure, don't theorize).